# PipeTwin · прогноз для двух реальных участков

По умолчанию ноутбук собирает предварительный `result.csv` для 1250022871 и 1250009583. Путь к файлу выводится под ячейкой сохранения. Следующая ячейка строит оба графика из этого CSV.

Если `target_date = None`, используется общая последняя дата в технологическом режиме. Укажите официальную дату T, когда она будет подтверждена. Базовый расчёт оценивает общую скорость по первому ВТД и не сопоставляет отдельные дефекты разных прогонов. Для сегментации последнего ВТД используется `Distance_adjusted`. Для участка без ВТД исходной толщиной служит паспортный номинал.

Для тестового примера задайте `demo_mode = True`. Чтобы использовать уже подготовленный прогноз, задайте его путь в `real_source`.


In [ ]:
from datetime import date
from importlib import reload
from io import StringIO
from pathlib import Path

from IPython.display import SVG, display
import matplotlib.pyplot as plt

from backend.corrosion import co2_rate, remaining_thickness
from backend.submission import result_csv
from backend import pipeline_data, forecasting, validation

# Refresh project modules when rerunning in an open kernel.
reload(pipeline_data)
reload(forecasting)
reload(validation)
PROJECT_ROOT = pipeline_data.PROJECT_ROOT
read_pipeline_csv = pipeline_data.read_pipeline_csv

demo_mode = False
target_date = None  # Или date(YYYY, MM, DD) после подтверждения даты T.
real_source = None  # None — рассчитать по данным проекта, Path(...) — собственный входной CSV.
source = PROJECT_ROOT / 'tests/result.csv' if demo_mode else (PROJECT_ROOT / real_source.expanduser() if real_source else None)
output = PROJECT_ROOT / ('result_demo.csv' if demo_mode else 'result.csv')
use_precomputed_result = True
submission_ids = None  # Например, ('ID_первого', 'ID_второго')
passport = PROJECT_ROOT / ('tests/passport.csv' if demo_mode else 'data/passport.csv')
years = 5
manual_rate = None
conditions = dict(
    temperature_c=30, total_pressure_bar=20, co2_mol_percent=0.5,
    ph=5.0, flow_velocity_ms=1.0, diameter_mm=200,
)

if source is None:
    predictions, run_info = forecasting.forecast_competition_pipelines(PROJECT_ROOT / 'data', target_date)
    pipes = predictions
    print(f"Дата прогноза: {run_info['target']}. ВТД: {run_info['first_vtd']} и {run_info['latest_vtd']}.")
    print(f"Базовая скорость: {run_info['rate']:.4f} мм/год; восстановлено сегментов: {run_info['filled_segments']}.")
else:
    if source.resolve() == output.resolve():
        raise ValueError('Укажите разные пути для входного и выходного CSV.')
    if not source.is_file():
        raise FileNotFoundError(
            f'Входной CSV не найден: {source}. Укажите существующий файл в real_source '
            'или включите demo_mode для проверки на тестовых данных.'
        )
    if passport and not passport.is_file():
        raise FileNotFoundError(f'Паспорт не найден: {passport}. Укажите существующий файл или passport = None.')
    pipes = read_pipeline_csv(source.read_bytes())
    if submission_ids is not None:
        by_id = {pipe['id']: pipe for pipe in pipes}
        missing = set(submission_ids) - by_id.keys()
        if missing:
            raise ValueError(f'Участки отсутствуют во входных данных: {sorted(missing)}')
        pipes = [by_id[pipe_id] for pipe_id in submission_ids]
    if len(pipes) != 2:
        raise ValueError('Выберите два участка для сдачи через submission_ids.')
    if any(pipe.get('coordinate_basis') == 'inspection_order' for pipe in pipes):
        raise ValueError('Сырые записи ВТД сначала нужно привязать к физическим сегментам.')
    lengths = {pipe['id']: pipe['length_m'] for pipe in read_pipeline_csv(passport.read_bytes())} if passport else {}


In [ ]:
if source is not None:
    import math

    rate = None
    if not use_precomputed_result:
        if not 0 < years <= 100:
            raise ValueError('Горизонт должен быть больше 0 и не более 100 лет.')
        rate = co2_rate(**conditions) if manual_rate is None else manual_rate
        if not math.isfinite(rate) or rate < 0:
            raise ValueError('Скорость должна быть конечной и неотрицательной.')
        print(f'Скорость утонения: {rate:.4g} мм/год')
    elif any(pipe.get('source_kind') == 'nominal' or any(row.get('replacement') for row in pipe['segments']) for pipe in pipes):
        raise ValueError('Паспорт или профиль с заменами требует расчёта. Укажите use_precomputed_result = False.')


    def thickness_for_result(segment):
        if rate is None:
            return segment['thickness_pred']
        wall = segment.get('replacement', {}).get('wall_mm', segment['thickness_pred'])
        return remaining_thickness(wall, rate, years)


    predictions = [
        {**pipe, 'length_m': lengths.get(pipe['id'], pipe.get('length_m')),
         'segments': [
             {'segment_id': row['segment_id'], 'thickness_pred': thickness_for_result(row)}
             for row in pipe['segments']
         ]}
        for pipe in pipes
    ]


In [ ]:
csv_text = result_csv(predictions)
output.write_bytes(csv_text.encode('utf-8'))
result = read_pipeline_csv(output.read_bytes())
if any(not pipe.get('length_m') for pipe in predictions):
    print('Сверьте количество сегментов с паспортом: для одного или обоих участков длина отсутствует.')
print(f'{output.resolve()}: {len(result)} участка, {sum(len(pipe["segments"]) for pipe in result)} строк.')

In [ ]:
fig, axes = plt.subplots(len(result), 1, figsize=(10, 3.3 * len(result)), layout='constrained')
names = {pipe['id']: pipe.get('name') for pipe in pipes}
for axis, pipe in zip(axes, result):
    distance = [segment['segment_id'] * 10 for segment in pipe['segments']]
    thickness = [segment['thickness_pred'] for segment in pipe['segments']]
    axis.plot(distance, thickness, color='#2563a8', marker='o' if len(distance) < 100 else None)
    title = names[pipe['id']] or pipe['id']
    axis.set(title=f'{title} · result.csv', xlabel='Начало сегмента, м', ylabel='Толщина, мм')
    axis.grid(alpha=0.25)
with StringIO() as buffer:
    fig.savefig(buffer, format='svg')
    display(SVG(data=buffer.getvalue()))
plt.close(fig)

## Историческая проверка

Для участка 1250022871 проверяем прогноз с 23.09.2018 на 06.07.2022. Толщины 2022 года используются только для оценки. Остаточная толщина есть в 508 проверочных сегментах, из них 321 имеют замеры в обоих ВТД. Для остальных начальная толщина восстановлена по соседним сегментам первого ВТД.

Строка «От ввода, без локальных замеров» иллюстрирует ограничение постоянного прогноза на проверочной трубе. Это не проверка участка 1250009583: контрольного ВТД для него в данных нет.

In [ ]:
if not demo_mode:
    report = validation.validate_baseline_forecast(PROJECT_ROOT / 'data')
    print(f"Проверка: {report['first']} → {report['held_out']}, {report['years']:.2f} года.")
    print(f"Оценено сегментов: {len(report['actual'])}, замеры в обоих ВТД: {report['measured_both']}.")
    for label, scores in report['metrics'].items():
        r2 = 'не определён' if scores['pearson_r2'] is None else f"{scores['pearson_r2']:.3f}"
        print(f"{label}: Pearson r² {r2}; MAE {scores['mae']:.3f} мм; RMSE {scores['rmse']:.3f} мм.")

    actual, predicted = report['actual'], report['predicted']
    low, high = min(min(actual), min(predicted)), max(max(actual), max(predicted))
    fig, ax = plt.subplots(figsize=(6.5, 5), layout='constrained')
    ax.scatter(actual, predicted, s=15, alpha=0.4, color='#2563a8')
    ax.plot([low, high], [low, high], '--', color='#64748b')
    ax.set(title='Историческая проверка · ВТД 2022',
           xlabel='Факт, мм', ylabel='Прогноз, мм', aspect='equal')
    ax.grid(alpha=0.2)
    with StringIO() as buffer:
        fig.savefig(buffer, format='svg')
        display(SVG(data=buffer.getvalue()))
    plt.close(fig)